# TS-Guessing reconstruction study

This notebook analyzes the local version-2 artifacts produced by the offline generator.
Set `RUN_DIR` and, optionally, `ANALYSIS_MODELS` below. Questions and prompts are in
Portuguese; explanations are in English. Analysis performs no inference and removes no
questions. Reconstruction rates are evidence for review, not contamination labels.


In [ ]:
from __future__ import annotations

import hashlib
import itertools
import json
import re
import sys
import unicodedata
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

RUN_DIR = None  # Set to output/<fingerprint>; relative paths resolve from repo root.
ANALYSIS_MODELS = None  # None: full recorded panel; otherwise an explicit list of model names.
BOOTSTRAP_REPLICATES = 2000
BOOTSTRAP_SEED = 42
PRIMARY_METRIC = "em_normalized"
CONDITIONS = ("without_source", "with_source")
METRICS = ("em_strict", "em_normalized", "em_label_tolerant", "token_f1",
           "empty_output", "copied_visible_choice", "length_limited")

cwd = Path.cwd().resolve()
REPO_ROOT = next(path for path in (cwd, *cwd.parents)
                 if (path / "pyproject.toml").exists() and (path / "ablations").exists())
STUDY_DIR = REPO_ROOT / "ablations/contamination/ts_guessing"
sys.path.insert(0, str(STUDY_DIR))
from artifacts import ARTIFACT_VERSION, read_json, read_jsonl

if RUN_DIR is None:
    available = sorted(path.parent for path in (STUDY_DIR / "output").glob("*/manifest.json")
                       if read_json(path).get("artifact_version") == ARTIFACT_VERSION)
    if len(available) != 1:
        raise ValueError(f"Set RUN_DIR explicitly. Available runs: {[str(p) for p in available]}")
    RUN_DIR = available[0]
else:
    RUN_DIR = Path(RUN_DIR).expanduser()
    if not RUN_DIR.is_absolute():
        RUN_DIR = REPO_ROOT / RUN_DIR
manifest = read_json(RUN_DIR / "manifest.json")
if manifest.get("artifact_version") != ARTIFACT_VERSION:
    raise ValueError(f"This notebook requires artifact_version={ARTIFACT_VERSION}; generate a new run.")
ANALYSIS_DIR = RUN_DIR / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")


def save_table(name, frame):
    frame.to_csv(ANALYSIS_DIR / f"{name}.csv", index=False)
    return frame


def save_records(name, rows):
    with (ANALYSIS_DIR / f"{name}.jsonl").open("w", encoding="utf-8") as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False) + "\n")


def save_figure(name, figure):
    figure.savefig(ANALYSIS_DIR / f"{name}.png", dpi=200, bbox_inches="tight")
    figure.savefig(ANALYSIS_DIR / f"{name}.pdf", bbox_inches="tight")


## 2. Snapshot, sample and execution coverage

The manifest identifies the exact input revision, configuration, software versions and
artifact checksums. `subject` and `macro_area` are preserved from the annotated dataset.
The unit of sampling is a question within `exam`, not an exam edition or knowledge area.
Area-specific results are descriptive of this exam-stratified sample.

Each task is identified by question, masked option and prompt condition. On resume, only
the latest result for each model-task pair is analyzed. Missing requests, context overflow
and request failures remain separate from incorrect reconstruction. A result terminated
by the output limit is a completed request with a diagnostic flag.


In [ ]:
for filename, key in (("sample.jsonl", "sample_sha256"), ("tasks.jsonl", "tasks_sha256")):
    if hashlib.sha256((RUN_DIR / filename).read_bytes()).hexdigest() != manifest[key]:
        raise ValueError(f"Artifact checksum mismatch: {filename}")
sample = pd.DataFrame(read_jsonl(RUN_DIR / "sample.jsonl"))
tasks = pd.DataFrame(read_jsonl(RUN_DIR / "tasks.jsonl"))
population = pd.DataFrame(manifest["exam_counts"])
population_weights = population.set_index("exam")["population"]
all_specs = manifest["config"]["models"]
selected = ANALYSIS_MODELS if ANALYSIS_MODELS is not None else [spec["name"] for spec in all_specs]
unknown = set(selected) - {spec["name"] for spec in all_specs}
if not selected or unknown or len(selected) != len(set(selected)):
    raise ValueError(f"Select a nonempty panel of unique recorded model names; unknown: {sorted(unknown)}")
model_specs = [spec for spec in all_specs if spec["name"] in selected]
model_names = [spec["name"] for spec in model_specs]
metadata_rows, response_rows, session_rows, batch_rows = [], [], [], []
for spec in model_specs:
    directory = RUN_DIR / "models" / spec["name"]
    metadata_path = directory / "metadata.json"
    metadata = json.loads(metadata_path.read_text()) if metadata_path.exists() else {}
    checkpoint = metadata.get("checkpoint", {})
    description = {**manifest["model_descriptions"][spec["name"]], **checkpoint}
    total = description.get("total_parameters")
    nominal = description.get("nominal_parameters_b")
    if metadata and (metadata["spec"] != spec or metadata["run_identity"] != manifest["identity"]):
        raise ValueError(f"Model metadata mismatch: {spec['name']}")
    metadata_rows.append({
        "model": spec["name"], "family": description.get("family", spec["name"]), "repo": spec["repo"],
        "max_model_len": metadata.get("server", {}).get("max_model_len", spec.get("max_model_len", manifest["config"]["server"]["max_model_len"])),
        "revision": description.get("revision"),
        "total_parameters_b": total / 1e9 if total else nominal,
        "parameters_source": "checkpoint metadata" if total else "model plan / nominal",
        **{key: description.get(key) for key in ("nominal_parameters_b", "active_parameters_b", "effective_parameters_b")},
        "execution_status": manifest["models"].get(spec["name"], {}).get("status", "not_started"),
    })
    latest = {row["task_id"]: row for row in read_jsonl(directory / "responses.jsonl")}
    for row in latest.values():
        usage = row.get("usage", {})
        response_rows.append({
            "task_id": row["task_id"], "model": spec["name"], "status": row["status"],
            "text": row.get("text", ""), "finish_reason": row.get("finish_reason"),
            "prompt_tokens": usage.get("prompt_tokens", 0),
            "completion_tokens": usage.get("completion_tokens", 0),
            **{key: row.get(key) for key in ("stage", "error", "session_id", "attempt", "batch_id", "rank")},
        })
    operations = read_jsonl(directory / "operations.jsonl")
    batches = [row for row in operations if row["kind"] == "batch"]
    attempts = [row for row in operations if row["kind"] == "attempt"]
    for row in operations:
        if row["kind"] != "session":
            continue
        session = {**row, "completed_requests": row["completed_tasks"]}
        for records, keys in ((attempts, ("startup_seconds", "processing_seconds")),
                              (batches, ("tokenization_seconds", "generation_seconds", "persistence_seconds"))):
            own = [item for item in records if item["session_id"] == row["session_id"]]
            session.update({key: sum(item.get(key, 0) for item in own) for key in keys})
        session_rows.append(session)
    batch_rows.extend({"model": spec["name"], **row} for row in batches)
models = pd.DataFrame(metadata_rows)
response_columns = ["task_id", "model", "status", "text", "finish_reason",
                    "prompt_tokens", "completion_tokens",
                    "stage", "error", "session_id", "attempt", "batch_id", "rank"]
responses = pd.DataFrame(response_rows, columns=response_columns)
task_columns = [column for column in tasks.columns if column not in ("prompt", "visible_choices", "target")]
grid = pd.MultiIndex.from_product([model_names, tasks.task_id], names=["model", "task_id"]).to_frame(index=False)
grid = grid.merge(tasks[task_columns], on="task_id", validate="many_to_one")
grid = grid.merge(responses, on=["model", "task_id"], how="left", validate="one_to_one")
grid["status"] = grid.status.fillna("not_run")
grid["subject"] = grid.subject.fillna("<missing>")
grid["macro_area"] = grid.macro_area.fillna("<missing>")
coverage = grid.groupby(["model", "condition", "status"]).size().unstack(fill_value=0).reset_index()
for status in ("completed", "context_overflow", "request_error", "not_run"):
    if status not in coverage:
        coverage[status] = 0
coverage["expected"] = coverage[["completed", "context_overflow", "request_error", "not_run"]].sum(axis=1)
coverage["completed_fraction"] = coverage.completed / coverage.expected
display(Markdown(f"**Run status:** {manifest['status']}; **population:** {manifest['population_size']:,}; "
                 f"**sample:** {manifest['sample_size']:,}; **tasks/model:** {manifest['tasks_per_model']:,}."))
display(population, models, save_table("request_coverage", coverage))
save_table("model_metadata", models)

# The sample retains the exact original row position before sampling.
catalog = [{**row, "source": manifest["dataset"], "source_snapshot_sha256": manifest["snapshot_hash"],
            "source_order_sha256": manifest["source_order_sha256"],
            "source_record_sha256": row["question_id"].split(":")[0],
            "run_identity": manifest["identity"]} for row in sample.to_dict("records")]
save_records("question_catalog", catalog)
panel = {"run_identity": manifest["identity"], "models": model_names,
         "families": models.groupby("family")["model"].apply(list).to_dict(),
         "metric": "em_normalized", "conditions": list(CONDITIONS),
         "weighting": "equal conditions, equal checkpoints within family, equal families",
         "eligibility": "all masks, both conditions, every selected model completed"}
(ANALYSIS_DIR / "panel.json").write_text(json.dumps(panel, ensure_ascii=False, indent=2) + "\n")
print("Analysis panel:", panel["families"])

preparation_rows = [row for row in read_jsonl(RUN_DIR / "operations.jsonl") if row["kind"] == "runtime_preparation"]


## 3. Reconstruction metrics

Strict exact match removes only outer whitespace. Normalized exact match additionally
applies Unicode NFKC and collapses whitespace; it preserves case, accents, numbers and
punctuation. The label-tolerant variant strips at most one leading label matching the
masked position. Explanations, quotes, other labels and semantic paraphrases are not
removed to manufacture matches.

Token F1 uses whitespace-delimited normalized tokens with multiplicity. It is a descriptive
overlap measure rather than a contamination decision. Empty expected strings are not counted
as successful reconstruction. Short and predictable targets require particular care.


In [ ]:
def normalize_text(text):
    return " ".join(unicodedata.normalize("NFKC", text).split())


def strip_response_label(text, label):
    pattern = rf"^\s*(?:\({label}\)|\[{label}\]|{label}\s*[.):\-])\s+"
    return re.sub(pattern, "", text, count=1).strip()


def score_response(row):
    prediction, target = row["text"].strip(), row["target"].strip()
    predicted, expected = normalize_text(prediction), normalize_text(target)
    tolerant = normalize_text(strip_response_label(prediction, row["mask_label"]))
    predicted_tokens, expected_tokens = Counter(predicted.split()), Counter(expected.split())
    overlap = sum((predicted_tokens & expected_tokens).values())
    denominator = sum(predicted_tokens.values()) + sum(expected_tokens.values())
    visible = [normalize_text(text) for text in row["visible_choices"]]
    return {
        "em_strict": float(bool(target) and prediction == target),
        "em_normalized": float(bool(expected) and predicted == expected),
        "em_label_tolerant": float(bool(expected) and tolerant == expected),
        "token_f1": 2 * overlap / denominator if denominator else 0.0,
        "empty_output": float(not prediction),
        "copied_visible_choice": float(bool(tolerant) and tolerant in visible),
        "length_limited": float(row["finish_reason"] == "length"),
    }


completed = grid.loc[grid.status.eq("completed")].copy()
completed = completed.merge(tasks[["task_id", "target", "visible_choices"]], on="task_id", validate="many_to_one")
metric_rows = [score_response(row) for row in completed.to_dict("records")]
scored = pd.concat([completed.reset_index(drop=True), pd.DataFrame(metric_rows, columns=METRICS)], axis=1)
scored["length_band"] = pd.cut(scored.target_words, bins=[-1, 0, 3, 7, np.inf],
                               labels=["empty", "1–3", "4–7", "≥8"])
save_columns = [column for column in scored.columns if column not in ("visible_choices",)]
save_table("task_scores", scored[save_columns])

question_keys = ["model", "condition", "question_id", "exam", "subject", "macro_area", "choice_count"]
question_coverage = grid.assign(is_completed=grid.status.eq("completed")).groupby(question_keys, dropna=False).agg(
    expected_masks=("task_id", "size"), completed_masks=("is_completed", "sum")
).reset_index()
question_means = scored.groupby(question_keys, dropna=False)[list(METRICS)].mean().reset_index()
question_any = scored.groupby(question_keys, dropna=False)[PRIMARY_METRIC].max().rename("any_reconstructed").reset_index()
questions = question_coverage.merge(question_means, on=question_keys, how="left").merge(
    question_any, on=question_keys, how="left")
questions["all_masks_completed"] = questions.completed_masks.eq(questions.expected_masks)
questions.loc[~questions.all_masks_completed, [*METRICS, "any_reconstructed"]] = np.nan
complete_questions = questions.loc[questions.all_masks_completed].copy()
save_table("question_scores_and_coverage", questions)
display(questions.groupby(["model", "condition"]).agg(
    sampled_questions=("question_id", "size"), complete_questions=("all_masks_completed", "sum")))

evidence = grid.merge(tasks[["task_id", "target", "visible_choices"]], on="task_id", validate="many_to_one")
evidence = evidence.merge(scored[["model", "task_id", *METRICS]], on=["model", "task_id"],
                          how="left", validate="one_to_one")
evidence = evidence.merge(sample[["question_id", "source_row_index"]], on="question_id", validate="many_to_one")
save_table("question_evidence", evidence.drop(columns="visible_choices"))
# JSONL preserves nested alternatives, null scores and exact IDs for future tooling.
(ANALYSIS_DIR / "question_evidence.jsonl").write_text(
    evidence.to_json(orient="records", lines=True, force_ascii=False), encoding="utf-8")


## 4. Question review and equal-family consensus

The ranking measures observed reconstruction. First average the distractor EMs within each
question/model/condition, then the two conditions per model, checkpoints within a family,
and finally families with equal weights. A family with six checkpoints has the same final
weight as a family with two. Changing the selected panel changes the estimand; `panel.json`
records the exact membership and weighting.

A question is eligible only if every mask in both conditions completed for every selected
model. Missing generations, context overflows and operational failures remain missing.
Incomplete questions have coverage records but no consensus score or rank. Family scores
also require all selected checkpoints in that family. Ties are ordered by `question_id`.

The catalog preserves original text, alternatives, annotations, source identity and the
zero-based original row position in the exact recorded input snapshot. Evidence rows
link each mask, target, response and diagnostic to
the original question. These exports support later review; this notebook removes no items.


In [ ]:
per_model = questions.groupby(["question_id", "model"], dropna=False).agg(
    completed_conditions=("all_masks_completed", "sum"),
    reconstruction_score=("em_normalized", "mean"),
).reset_index()
per_model["model_complete"] = per_model.completed_conditions.eq(len(CONDITIONS))
per_model.loc[~per_model.model_complete, "reconstruction_score"] = np.nan
per_model = per_model.merge(models[["model", "family"]], on="model", validate="many_to_one")
save_table("question_model_scores", per_model)

family_sizes = models.groupby("family").size().rename("expected_models")
family_scores = per_model.groupby(["question_id", "family"]).agg(
    completed_models=("model_complete", "sum"), family_score=("reconstruction_score", "mean"),
).reset_index().merge(family_sizes, on="family", validate="many_to_one")
family_scores["family_complete"] = family_scores.completed_models.eq(family_scores.expected_models)
family_scores.loc[~family_scores.family_complete, "family_score"] = np.nan
save_table("question_family_scores", family_scores)

review = family_scores.groupby("question_id").agg(
    complete_families=("family_complete", "sum"), complete_models=("completed_models", "sum"),
    consensus_score=("family_score", "mean"),
).reindex(sample.question_id).reset_index()
review["expected_models"] = len(model_names)
review["expected_families"] = len(family_sizes)
review[["complete_models", "complete_families"]] = review[["complete_models", "complete_families"]].fillna(0).astype(int)
review["model_coverage"] = review.complete_models / review.expected_models
review["eligible"] = review.complete_models.eq(review.expected_models) & review.complete_families.eq(review.expected_families)
review.loc[~review.eligible, "consensus_score"] = np.nan

# Describe predictable distractors once per mask, independently of model coverage.
masks = tasks[tasks.condition.eq("without_source")].assign(short_target=tasks.target_words.between(1, 3))
diagnostics = masks.groupby("question_id").agg(
    distractors=("task_id", "size"), short_targets=("short_target", "sum"),
    numeric_targets=("target_numeric", "sum"), targets_in_context=("target_in_context", "sum"),
).reset_index()
locators = ["question_id", "source_row_index", "exam", "exam_edition", "num", "subject", "macro_area"]
review = review.merge(sample[locators], on="question_id", validate="one_to_one").merge(
    diagnostics, on="question_id", how="left", validate="one_to_one")
review = review.merge(family_scores.pivot(index="question_id", columns="family", values="family_score").add_prefix("family_score_"),
                      on="question_id", how="left", validate="one_to_one")
ranking = review[review.eligible].sort_values(["consensus_score", "question_id"], ascending=[False, True]).copy()
ranking.insert(0, "rank", np.arange(1, len(ranking) + 1))
review = review.merge(ranking[["question_id", "rank"]], on="question_id", how="left", validate="one_to_one")
review["rank"] = review["rank"].astype("Int64")
save_table("question_review_coverage", review)
save_table("question_ranking", ranking)
for name, frame in (("question_review_coverage", review), ("question_ranking", ranking)):
    (ANALYSIS_DIR / f"{name}.jsonl").write_text(frame.to_json(orient="records", lines=True, force_ascii=False), encoding="utf-8")
print(f"Eligible for consensus: {len(ranking):,}/{len(sample):,} questions; {len(family_sizes)} equally weighted families.")
display(ranking.head(30), review.loc[~review.eligible].head(10))
if complete_questions.empty:
    raise ValueError("No complete questions. Catalog, evidence and consensus coverage were exported; inspect operational logs.")


## 5. Question-level aggregation and uncertainty

The primary metric is **normalized EM averaged over all distractors of each question**,
then over questions. A question enters the primary analysis only if all its distractor
requests completed for that model-condition pair. This keeps four- and five-option questions
equally weighted and prevents incomplete easy subsets from silently replacing a question.
The fraction with any reconstructed distractor is secondary and separated by choice count.

Bootstrap intervals use 2,000 resamples of questions within exam. Distractors remain
together because they are not independent observations. Comparisons reuse the same resampled
questions on both sides. Sample, exam-macro and population-weighted estimates are distinct.
The full-population weighted estimate is left undefined if any exam has no complete questions.
Intervals describe variability over these questions; they do not calibrate a detector's
false-positive rate or account for uncertainty about training exposure.


In [ ]:
def bootstrap_interval(frame, metric, weighting="sample"):
    groups = [(exam, group[metric].dropna().to_numpy(dtype=float))
              for exam, group in frame.groupby("exam", sort=True)]
    groups = [(exam, values) for exam, values in groups if len(values)]
    if not groups:
        return np.nan, np.nan, np.nan
    if weighting == "population" and {exam for exam, _ in groups} != set(population_weights.index):
        return np.nan, np.nan, np.nan
    if weighting == "population":
        weights = np.array([population_weights[exam] for exam, _ in groups], dtype=float)
    elif weighting == "macro":
        weights = np.ones(len(groups))
    else:
        weights = np.array([len(values) for _, values in groups], dtype=float)
    weights /= weights.sum()
    estimate = sum(weight * values.mean() for weight, (_, values) in zip(weights, groups))
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    draws = np.zeros(BOOTSTRAP_REPLICATES)
    for weight, (_, values) in zip(weights, groups):
        for start in range(0, BOOTSTRAP_REPLICATES, 128):
            stop = min(start + 128, BOOTSTRAP_REPLICATES)
            indices = rng.integers(0, len(values), size=(stop - start, len(values)))
            draws[start:stop] += weight * values[indices].mean(axis=1)
    lower, upper = np.quantile(draws, [0.025, 0.975])
    return estimate, lower, upper


summary_rows = []
for (model, condition), group in complete_questions.groupby(["model", "condition"]):
    row = {"model": model, "condition": condition, "complete_questions": len(group),
           "covered_exams": group.exam.nunique(),
           "question_coverage": len(group) / manifest["sample_size"]}
    row.update({metric: group[metric].mean() for metric in METRICS})
    for weighting in ("sample", "macro", "population"):
        point, lower, upper = bootstrap_interval(group, PRIMARY_METRIC, weighting)
        row.update({f"{weighting}_em": point, f"{weighting}_lower": lower, f"{weighting}_upper": upper})
    summary_rows.append(row)
summary = pd.DataFrame(summary_rows).merge(models, on="model", validate="many_to_one")
display(save_table("overall_summary", summary))

subgroup_tables = {}
for field in ("exam", "subject", "macro_area"):
    expected = question_coverage.groupby(["model", "condition", field], dropna=False).size().rename("sampled_questions")
    observed = complete_questions.groupby(["model", "condition", field], dropna=False).agg(
        complete_questions=("question_id", "size"), mean_em=(PRIMARY_METRIC, "mean"))
    table = expected.to_frame().join(observed).reset_index()
    table["complete_questions"] = table.complete_questions.fillna(0).astype(int)
    table["coverage"] = table.complete_questions / table.sampled_questions
    subgroup_tables[field] = save_table(f"by_{field}", table)
display(subgroup_tables["exam"], subgroup_tables["macro_area"])


## 6. Paired source-hint and model comparisons

Each contrast uses the intersection of fully observed questions, with the same masks
on both sides. Positive source-hint differences mean greater reconstruction after exam
identification. They do not demonstrate ingestion of that exam into training.

Model contrasts are descriptive: family, generation, architecture and training data differ.
Family labels do not imply independent architectures: Tucano2 uses Qwen architecture.
Nemotron checkpoints can span different lineages and dense/hybrid MoE architectures;
inspect recorded metadata. Differences are not a controlled size-only contrast.
Confidence intervals are pointwise; the many contrasts are exploratory and are not
multiple-testing-adjusted discoveries of contamination. The comparison table includes
the intersection size, covered exams and full-population estimate when identifiable.


In [ ]:
def paired_contrast(left, right, left_name, right_name, condition):
    shared = left[["question_id", "exam", PRIMARY_METRIC]].merge(
        right[["question_id", PRIMARY_METRIC]], on="question_id", suffixes=("_left", "_right"),
        validate="one_to_one")
    if shared.empty:
        return None
    shared["difference"] = shared[f"{PRIMARY_METRIC}_right"] - shared[f"{PRIMARY_METRIC}_left"]
    result = {"left": left_name, "right": right_name, "condition": condition,
              "shared_questions": len(shared), "covered_exams": shared.exam.nunique()}
    for weighting in ("sample", "population"):
        point, lower, upper = bootstrap_interval(shared, "difference", weighting)
        result.update({f"{weighting}_difference": point, f"{weighting}_lower": lower,
                       f"{weighting}_upper": upper})
    return result


hint_rows, model_rows = [], []
for model, group in complete_questions.groupby("model"):
    result = paired_contrast(group[group.condition.eq("without_source")],
                             group[group.condition.eq("with_source")],
                             "without_source", "with_source", "source_hint")
    if result:
        hint_rows.append({"model": model, **result})
for condition in CONDITIONS:
    frames = {model: group for model, group in
              complete_questions[complete_questions.condition.eq(condition)].groupby("model")}
    for left_name, right_name in itertools.combinations(sorted(frames), 2):
        result = paired_contrast(frames[left_name], frames[right_name], left_name, right_name, condition)
        if result:
            model_rows.append(result)
hint_effects = save_table("paired_source_hint_effects", pd.DataFrame(hint_rows))
model_effects = save_table("paired_model_effects", pd.DataFrame(model_rows))
display(hint_effects, model_effects)

# Equal-question panel for plots; inactive models are explicitly listed below.
common_rows = []
for condition in CONDITIONS:
    group = complete_questions[complete_questions.condition.eq(condition)]
    active = sorted(group.model.unique())
    print(condition, "models with complete questions:", active)
    print("Excluded from this plot:", sorted(set(model_names) - set(active)))
    if not active:
        continue
    shared_ids = set.intersection(*(set(group.loc[group.model.eq(model), "question_id"]) for model in active))
    for model in active:
        common = group[group.model.eq(model) & group.question_id.isin(shared_ids)]
        point, lower, upper = bootstrap_interval(common, PRIMARY_METRIC)
        common_rows.append({"model": model, "condition": condition,
                            "shared_questions": len(common), "mean_em": point,
                            "lower": lower, "upper": upper})
common_summary = pd.DataFrame(common_rows).merge(models, on="model", validate="many_to_one")
save_table("common_question_panel", common_summary)
display(common_summary)


## 7. Distractor predictability and presentation diagnostics

Length bands, numeric alternatives and targets already present in visible context help
identify readily reconstructible alternatives. These diagnostics do not establish which
items were unseen. Distractor-level tables are secondary; they do not replace the
question-level primary metric. All masks are tested, so the probability of at least one
match depends on the number of distractors and is shown separately for four/five choices.


In [ ]:
length_results = scored.groupby(["model", "condition", "length_band"], observed=True).agg(
    completed_masks=("task_id", "size"), questions=("question_id", "nunique"),
    mean_em=(PRIMARY_METRIC, "mean"), label_tolerant_em=("em_label_tolerant", "mean"),
    token_f1=("token_f1", "mean")).reset_index()
flag_tables = []
for flag in ("target_numeric", "target_in_context"):
    table = scored.groupby(["model", "condition", flag]).agg(
        completed_masks=("task_id", "size"), questions=("question_id", "nunique"),
        mean_em=(PRIMARY_METRIC, "mean")).reset_index().rename(columns={flag: "flag_value"})
    table["diagnostic"] = flag
    flag_tables.append(table)
any_results = complete_questions.groupby(["model", "condition", "choice_count"]).agg(
    complete_questions=("question_id", "size"), any_reconstruction_rate=("any_reconstructed", "mean")
).reset_index()
display(save_table("distractor_length", length_results),
        save_table("predictability_flags", pd.concat(flag_tables, ignore_index=True)),
        save_table("any_reconstruction_by_choice_count", any_results))


## 8. Figures

Exam/area heatmaps are descriptive and accompany the exported coverage tables. Missing
cells remain missing rather than zero. The model-size plot uses a common question panel
within each prompt condition; the common panels can differ between conditions. Use the
paired source-hint table to compare prompts. Sizes are total checkpoint parameters, with
effective-parameter counts for Gemma E2B and Nemotron 3 Nano separately reported
in the metadata table. Nominal size labels can differ from stored parameter counts.


In [ ]:
for field in ("exam", "macro_area"):
    for condition in CONDITIONS:
        table = subgroup_tables[field]
        subset = table[table.condition.eq(condition)]
        matrix = subset.pivot(index="model", columns=field, values="mean_em").reindex(model_names).dropna(how="all")
        if matrix.empty:
            continue
        fig, ax = plt.subplots(figsize=(max(9, matrix.shape[1] * 0.65), max(4, len(matrix) * 0.5)))
        sns.heatmap(matrix, annot=True, fmt=".3f", vmin=0, vmax=1, cmap="Blues", ax=ax,
                    cbar_kws={"label": "Question-mean normalized EM"})
        ax.set_title(f"{field}: {condition} (see coverage table)")
        save_figure(f"heatmap_{field}_{condition}", fig)
        plt.show()

plot_data = common_summary.dropna(subset=["mean_em"])
if not plot_data.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    palette = dict(zip(sorted(models.family.unique()), sns.color_palette("tab10", models.family.nunique())))
    for (family, condition), group in plot_data.groupby(["family", "condition"]):
        group = group.sort_values("total_parameters_b")
        ax.errorbar(group.total_parameters_b, group.mean_em,
                    yerr=np.maximum(0, np.vstack([group.mean_em - group.lower, group.upper - group.mean_em])),
                    marker="o" if condition == "without_source" else "s", capsize=3,
                    linestyle="-" if condition == "without_source" else "--",
                    color=palette[family], label=f"{family} / {condition}")
    ax.set(xscale="log", xlabel="Total checkpoint parameters (billions)",
           ylabel="Question-mean normalized EM", title="Common-question panel within each condition")
    ax.legend(fontsize=8)
    save_figure("model_size", fig)
    plt.show()

if not hint_effects.empty:
    table = hint_effects.sort_values("model").reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(9, max(4, len(table) * 0.45)))
    ax.errorbar(table.sample_difference, np.arange(len(table)),
                xerr=np.vstack([table.sample_difference - table.sample_lower,
                                table.sample_upper - table.sample_difference]), fmt="o", capsize=3)
    ax.set_yticks(np.arange(len(table)), table.model)
    ax.axvline(0, color="gray", linestyle="--")
    ax.set(xlabel="With source − without source: normalized EM", title="Paired source-identification effect")
    save_figure("source_hint_effect", fig)
    plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for ax, condition in zip(axes, CONDITIONS):
    subset = length_results[length_results.condition.eq(condition)]
    sns.lineplot(data=subset, x="length_band", y="mean_em", hue="model", marker="o", ax=ax)
    ax.set(title=condition, xlabel="Distractor words", ylabel="Distractor-level normalized EM")
    ax.legend(fontsize=7)
save_figure("distractor_length", fig)
plt.show()


## 9. Observed computational cost

Session wall time includes initialization, shutdown and retries. Processing wall time,
recorded per attempt, is the denominator for tasks/s and output tokens/s. Tokenization
and generation times are sums across rank batches and can exceed processing wall time
with DP > 1. Persistence is measured by the sole response writer. Batch records include
serialization and batch wall time; synchronization and queue transfer also contribute.

No individual task latency is measured. Batch duration must not be presented as individual
latency. Auxiliary MoE requests are excluded from task and token totals while their
overhead remains in runtime cost. Incomplete batches retain the measurements available.
GPU-hours are estimated from session wall time times DP × TP, not GPU utilization.

`operations.jsonl` contains launch, execution, initialization, batch, error, attempt and
session records. It also preserves completed work before failures. Isolated runtime
preparation is recorded at the run level and reported separately from model execution.


In [ ]:
cost_rows = []
runtime_preparation = pd.DataFrame(preparation_rows)
if not runtime_preparation.empty:
    print("Isolated runtime preparation time (seconds):", runtime_preparation.elapsed_seconds.sum())
    save_table("runtime_preparation", runtime_preparation)
if batch_rows:
    batches = save_table("batch_timings", pd.DataFrame(batch_rows))
    display(batches.groupby(["model", "rank"])[
        ["tokenization_seconds", "generation_seconds", "persistence_seconds"]
    ].sum())
for spec in model_specs:
    sessions = [row for row in session_rows if row["model"] == spec["name"]]
    output = responses[responses.model.eq(spec["name"]) & responses.status.eq("completed")]
    wall = sum(row.get("wall_seconds", 0) for row in sessions)
    processing = sum(row["processing_seconds"] for row in sessions)
    completed_count = sum(row.get("completed_requests", 0) for row in sessions)
    tokens = sum(row.get("completion_tokens", 0) for row in sessions)
    tp = spec.get("tensor_parallel_size", manifest["config"]["server"]["tensor_parallel_size"])
    dp = spec.get("data_parallel_size", manifest["config"]["server"].get("data_parallel_size", 1))
    per_replica_concurrency = spec.get("concurrency", manifest["config"]["server"]["concurrency"])
    gpu_hours = sum(row.get("wall_seconds", 0) * row.get("gpu_count", dp * tp)
                    for row in sessions) / 3600
    cost_rows.append({
        "model": spec["name"], "sessions": len(sessions), "wall_seconds": wall,
        "startup_seconds": sum(row.get("startup_seconds", 0) for row in sessions),
        "processing_wall_seconds": processing, "estimated_gpu_hours": gpu_hours,
        "rank_tokenization_seconds": sum(row["tokenization_seconds"] for row in sessions),
        "rank_generation_seconds": sum(row["generation_seconds"] for row in sessions),
        "persistence_seconds": sum(row["persistence_seconds"] for row in sessions),
        "data_parallel_size": dp, "tensor_parallel_size": tp, "configured_gpu_count": dp * tp,
        "concurrency_per_replica": per_replica_concurrency,
        "total_concurrency": per_replica_concurrency * dp,
        "batch_size_per_replica": spec.get("batch_size", manifest["config"]["server"]["batch_size"]),
        "session_completed_requests": completed_count, "unique_completed_tasks": len(output),
        "prompt_tokens": sum(row.get("prompt_tokens", 0) for row in sessions),
        "completion_tokens": tokens,
        "requests_per_second": completed_count / processing if processing else np.nan,
        "output_tokens_per_second": tokens / processing if processing else np.nan,
    })
costs = save_table("observed_cost", pd.DataFrame(cost_rows))
display(costs)
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=costs[costs.sessions.gt(0)], x="model", y="wall_seconds", ax=ax)
ax.tick_params(axis="x", rotation=60)
ax.set(title="Observed runtime wall time, including resumed sessions", ylabel="Seconds", xlabel="Model")
save_figure("observed_wall_time", fig)
plt.show()


## 10. Interpretation and limitations

- These observations are reconstruction rates, not a calibrated contamination prevalence.
- Sampling only 10% of each exam limits item-level conclusions about the remaining dataset.
- All distractors are tested, but they differ in predictability and are not independent.
- Source hints can improve retrieval or alter instruction following; their effect is not
  a controlled estimate of training exposure.
- Family and size differences confound architecture, generation, language specialization,
  training data and post-training. Llama 1B/3B are version 3.2; 8B is version 3.1.
  The current model plan includes dense and MoE Qwen/Gemma variants; Gemma E2B/E4B
  use PLE. Nemotron Nano 4B is compressed from a different lineage than the 30B-A3B
  MoE. These are descriptive architecture comparisons, not controlled scaling curves.
  The current panel disables native thinking via `enable_thinking=false` where supported.
- Greedy decoding controls sampling randomness but does not guarantee identical floating-point
  behavior across GPU, batching or library changes. Revisions and software versions are recorded.
- Complete-question filtering and common intersections can change coverage. Tables expose
  the resulting population; small areas and missing exams constrain interpretation.
- Without exposure labels or controlled contamination experiments, precision/recall of
  contamination detection cannot be inferred from these matches.
- No dataset questions are removed, and no model receives a binary contamination verdict.

## 11. Run-specific findings

The following summary is derived from the loaded artifacts. Inspect the paired contrasts
and predictability diagnostics before assigning meaning to differences between models.


In [ ]:
lines = [
    "# TS-Guessing analysis", f"Run: {RUN_DIR.name}",
    f"Generation status: {manifest['status']}",
    f"Sample: {manifest['sample_size']} questions; {manifest['tasks_per_model']} tasks/model.",
    f"Models with completed responses: {scored.model.nunique()} / {len(model_names)}.",
    f"Equal-family consensus ranking: {len(ranking):,}/{len(sample):,} eligible questions.",
    f"Selected panel: {len(model_names)} models in {len(family_sizes)} equally weighted families.",
    "Rates describe distractor reconstruction; they do not certify training exposure.",
]
for row in summary.itertuples():
    lines.append(f"{row.model} / {row.condition}: normalized EM {row.sample_em:.4f} "
                 f"(95% CI {row.sample_lower:.4f}–{row.sample_upper:.4f}); "
                 f"complete-question coverage {row.question_coverage:.1%}.")
if manifest["status"] != "completed":
    lines.append("The generation is incomplete. Comparisons apply only to the documented intersections.")
report = "\n\n".join(lines)
(ANALYSIS_DIR / "findings.md").write_text(report + "\n", encoding="utf-8")
display(Markdown(report))
